# PHASE 6 - XGBoost

This notebook trains an XGBoost regressor on CPU using the chronological splits from PHASE 3. A small validation search selects hyperparameters before the final test evaluation.

GPU is disabled explicitly with `tree_method="hist"` and `device="cpu"`. The test split is not used during hyperparameter selection.

In [ ]:
from pathlib import Path
from time import perf_counter
import pickle

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

def find_project_root():
    for candidate in [Path("."), Path("..")]:
        if (candidate / "data/processed/train.csv").exists():
            return candidate
    raise FileNotFoundError("Run PHASE 3 first to create data/processed/train.csv.")

project_root = find_project_root()
processed_dir = project_root / "data/processed"
metrics_dir = project_root / "results/metrics"
figures_dir = project_root / "results/figures"
models_dir = project_root / "models"
for directory in [metrics_dir, figures_dir, models_dir]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_split(name):
    return pd.read_csv(processed_dir / f"{name}.csv", parse_dates=["timestamp", "dteday"])

train_df = load_split("train")
validation_df = load_split("validation")
test_df = load_split("test")

target_column = "cnt"
excluded_columns = {target_column, "instant", "dteday", "timestamp", "casual", "registered"}
feature_columns = [column for column in train_df.columns if column not in excluded_columns]
X_train, y_train = train_df[feature_columns], train_df[target_column]
X_validation, y_validation = validation_df[feature_columns], validation_df[target_column]
X_test, y_test = test_df[feature_columns], test_df[target_column]

assert train_df["timestamp"].max() < validation_df["timestamp"].min()
assert validation_df["timestamp"].max() < test_df["timestamp"].min()
print(f"Project root: {project_root.resolve()}")
print(f"Features: {len(feature_columns)}")
print(f"Rows train/validation/test: {len(train_df)}/{len(validation_df)}/{len(test_df)}")

## CPU-friendly validation search

In [ ]:
candidates = [
    {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "min_child_weight": 1},
    {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.05, "min_child_weight": 1},
    {"n_estimators": 400, "max_depth": 6, "learning_rate": 0.03, "min_child_weight": 3},
]

def calculate_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": mean_squared_error(y_true, predictions) ** 0.5,
        "R2": r2_score(y_true, predictions),
    }

search_results = []
for parameters in candidates:
    candidate_model = XGBRegressor(
        **parameters,
        objective="reg:squarederror",
        tree_method="hist",
        device="cpu",
        n_jobs=-1,
        random_state=42,
    )
    start_time = perf_counter()
    candidate_model.fit(X_train, y_train)
    elapsed = perf_counter() - start_time
    validation_predictions = candidate_model.predict(X_validation)
    search_results.append({**parameters, **calculate_metrics(y_validation, validation_predictions), "training_time": elapsed})

search_results_df = pd.DataFrame(search_results).sort_values("RMSE").reset_index(drop=True)
display(search_results_df.round(4))
best_parameters = search_results_df.iloc[0][["n_estimators", "max_depth", "learning_rate", "min_child_weight"]].to_dict()
best_parameters["n_estimators"] = int(best_parameters["n_estimators"])
best_parameters["max_depth"] = int(best_parameters["max_depth"])
best_parameters["min_child_weight"] = int(best_parameters["min_child_weight"])
print("Selected parameters:", best_parameters)

In [ ]:
model = XGBRegressor(
    **best_parameters,
    objective="reg:squarederror",
    tree_method="hist",
    device="cpu",
    n_jobs=-1,
    random_state=42,
)
start_time = perf_counter()
model.fit(X_train, y_train)
training_time = perf_counter() - start_time
validation_predictions = model.predict(X_validation)
test_predictions = model.predict(X_test)

metrics = pd.DataFrame([
    {"model": "XGBoost", "split": "validation", **calculate_metrics(y_validation, validation_predictions), "Training Time": training_time},
    {"model": "XGBoost", "split": "test", **calculate_metrics(y_test, test_predictions), "Training Time": training_time},
])
display(metrics.round(4))
metrics.to_csv(metrics_dir / "xgboost_metrics.csv", index=False)
search_results_df.to_csv(metrics_dir / "xgboost_validation_search.csv", index=False)

In [ ]:
importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
importance.to_csv(metrics_dir / "xgboost_feature_importance.csv", index=False)

plt.figure(figsize=(10, 8))
top_importance = importance.head(15).sort_values("importance")
plt.barh(top_importance["feature"], top_importance["importance"])
plt.title("XGBoost: top 15 feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(figures_dir / "xgboost_feature_importance.png", dpi=150)
plt.show()
display(importance.head(15))

In [ ]:
comparison = pd.DataFrame({
    "timestamp": test_df["timestamp"],
    "actual": y_test.to_numpy(),
    "predicted": test_predictions,
})
plt.figure(figsize=(14, 5))
plt.plot(comparison["timestamp"], comparison["actual"], label="Actual", linewidth=1)
plt.plot(comparison["timestamp"], comparison["predicted"], label="Predicted", linewidth=1)
plt.title("XGBoost: actual vs predicted demand on test set")
plt.xlabel("Time")
plt.ylabel("Bike rentals (cnt)")
plt.legend()
plt.tight_layout()
plt.savefig(figures_dir / "xgboost_actual_vs_predicted.png", dpi=150)
plt.show()

model_path = models_dir / "xgboost.pkl"
with model_path.open("wb") as model_file:
    pickle.dump({"model": model, "feature_columns": feature_columns}, model_file)
print(f"Saved model to: {model_path}")

## Phase 6 conclusion

The selected CPU-only XGBoost model is evaluated on the untouched test set and can be compared with the Linear Regression and Random Forest baselines.